# 6.1 Klasifikasi Sawah vs Non-Sawah (Sentinel-2A Kamal)

**Supervised learning** (berbeda dari clustering Bab 5 yang unsupervised): model belajar dari label sampel.

**Status 2026-10-03: SELESAI (level kandidat).** Training memakai `data/spasial/kamal_sawah.shp` (100 poligon: 50 `sawah` + 50 `non_sawah`, CRS EPSG:32749) — turunan yang dinormalisasi dari `data/digitasi_kamal/sample_polygon_100_kamal.shp`. PROVENANSI JUJUR: sampel ini kandidat semi-otomatis (K-Means + interpretasi visual analis, lihat `data/digitasi_kamal/klasifikasi_kamal.py`), BUKAN digitasi manual QGIS + survei lapangan. Skor sempurna pada test set mencerminkan sirkularitas itu, bukan akurasi lapangan.

In [1]:
import os
import numpy as np
import pandas as pd

SEED = 42
SHP = "../data/spasial/kamal_sawah.shp"
RASTER = "../data/spasial/s2_kamal.tif"
OUT = "../data/processed"
os.makedirs(OUT, exist_ok=True)

DATA_OK = os.path.exists(SHP) and os.path.exists(RASTER)
print("shapefile:", "ADA" if os.path.exists(SHP) else "BELUM TERSEDIA")
print("raster:", "ADA" if os.path.exists(RASTER) else "BELUM TERSEDIA")

shapefile: ADA
raster: ADA


## Spesifikasi data (MANUAL_CHECKLIST.txt butir [1])

- Shapefile: `data/spasial/kamal_sawah.shp` — 100 poligon (50 `sawah` + 50 `non_sawah`), kolom `id` unik (`S001..S050`, `N001..N050`) + `kelas` (+`luas`). CRS EPSG:32749, sama dengan raster (tanpa reproyeksi).
- Sumber geometri: `data/digitasi_kamal/sample_polygon_100_kamal.shp` (CRS sama); label Sawah/Non-Sawah dinormalisasi menjadi sawah/non_sawah; id integer menjadi kode S/N. Sel ekstraksi juga menormalisasi ulang saat baca (tahan varian kapital/strip) agar rerun aman.
- Raster: `data/spasial/s2_kamal.tif` (komposit median 2026-08-05 s.d. 2026-08-26, 6 band B02/B03/B04/B08/B11/B12, 875x865, EPSG:32749, int16, nodata -32768).
- Fitur per poligon = **mean tiap band** di dalam poligon (piksel nodata dikecualikan).

In [2]:
import geopandas as gpd
import rasterio
from rasterio.mask import mask

if not DATA_OK:
    print("STATUS: BELUM TERSEDIA — pipeline dilewati. Isi sel ini akan aktif saat data tiba.")
    df_feat = None
else:
    shp = gpd.read_file(SHP)
    shp["kelas"] = shp["kelas"].astype(str).str.strip().str.lower().str.replace("-", "_")
    _vals = set(shp["kelas"].unique())
    assert _vals <= {"sawah", "non_sawah"}, "nilai kelas tak valid"
    assert shp["id"].is_unique, "kolom id tidak unik!"
    assert shp.geometry.is_valid.all() and (~shp.geometry.is_empty).all(), "ada geometry invalid/kosong!"
    src = rasterio.open(RASTER)
    if str(shp.crs) != str(src.crs):
        shp = shp.to_crs(src.crs)
        print("CRS shapefile direproyeksi ke CRS raster:", src.crs)
    nodata = src.nodata
    rows = []
    for _, r in shp.iterrows():
        img, _ = mask(src, [r.geometry], crop=True)
        img = img.astype(float)
        img[img == nodata] = np.nan
        means = [float(np.nanmean(img[b])) for b in range(src.count)]
        rows.append({"id": r["id"], "kelas": r["kelas"], **{f"band_{i+1}": v for i, v in enumerate(means)}})
    df_feat = pd.DataFrame(rows).dropna()
    print(df_feat["kelas"].value_counts().to_dict(), "sampel:", len(df_feat))
    df_feat.to_csv(f"{OUT}/sawah_features.csv", index=False)
    print("tersimpan: data/processed/sawah_features.csv")


{'non_sawah': 50, 'sawah': 50} sampel: 100
tersimpan: data/processed/sawah_features.csv


## Split, training, evaluasi

Split stratified 80/20 dengan seed tetap. Model: RandomForest (300 pohon, seed 42). Metrik: confusion matrix, accuracy, precision, recall, F1 + CV 5-fold.

**Keterbatasan jujur (wajib dibaca):** (1) Label training adalah kandidat semi-otomatis turunan K-Means pada raster yang sama sehingga skor mendekati sempurna karena sirkular — bukan bukti akurasi lapangan. (2) Training set hanya berisi sawah/pohon-kebun/terbangun (kuota 50/25/25; kelas air/tambak kode 4 TIDAK disampel) sehingga peta biner tidak dapat dipercaya pada area air/tambak. (3) Train/test bisa berdekatan spasial — idealnya split blok spasial bila sampel bertambah. Validasi lapangan/citra resolusi tinggi tetap diperlukan sebelum dipakai.

In [3]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, classification_report

if df_feat is None:
    print("STATUS: BELUM TERSEDIA — training dilewati.")
else:
    X = df_feat.filter(like="band_").to_numpy()
    y = (df_feat["kelas"] == "sawah").to_numpy().astype(int)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
    clf = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1)
    clf.fit(Xtr, ytr)
    pred = clf.predict(Xte)
    cm = confusion_matrix(yte, pred)
    print("confusion matrix (baris=aktual, kolom=prediksi [non_sawah, sawah]):")
    print(cm)
    print(classification_report(yte, pred, target_names=["non_sawah", "sawah"]))
    metrics = {"accuracy": accuracy_score(yte, pred),
               "precision_sawah": precision_score(yte, pred, zero_division=0),
               "recall_sawah": recall_score(yte, pred, zero_division=0),
               "f1_sawah": f1_score(yte, pred, zero_division=0)}
    print(metrics)
    cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
    s = cross_val_score(clf, X, y, cv=cv)
    print("cv5 accuracy:", s.round(3).tolist(), "mean:", round(float(s.mean()), 4))
    print("feature importance (band_1..6 = B02,B03,B04,B08,B11,B12):", clf.feature_importances_.round(3).tolist())
    pd.DataFrame([metrics]).to_csv(f"{OUT}/sawah_metrics.csv", index=False)
    import joblib
    joblib.dump(clf, f"{OUT}/sawah_rf.joblib")
    print("tersimpan: sawah_metrics.csv + sawah_rf.joblib")


confusion matrix (baris=aktual, kolom=prediksi [non_sawah, sawah]):
[[10  0]
 [ 0 10]]
              precision    recall  f1-score   support

   non_sawah       1.00      1.00      1.00        10
       sawah       1.00      1.00      1.00        10

    accuracy                           1.00        20
   macro avg       1.00      1.00      1.00        20
weighted avg       1.00      1.00      1.00        20

{'accuracy': 1.0, 'precision_sawah': 1.0, 'recall_sawah': 1.0, 'f1_sawah': 1.0}
cv5 accuracy: [1.0, 1.0, 1.0, 1.0, 1.0] mean: 1.0
feature importance (band_1..6 = B02,B03,B04,B08,B11,B12): [0.218, 0.165, 0.205, 0.151, 0.085, 0.175]
tersimpan: sawah_metrics.csv + sawah_rf.joblib


## Peta hasil klasifikasi

Raster Sentinel-2 ke fitur per piksel ke prediksi model ke raster terklasifikasi (`sawah_classified.tif`, 1 = sawah, 0 = non-sawah, 255 = nodata) + pratinjau PNG. Ini **peta prediksi biner**, berbeda dari peta clustering Bab 5 (kelompok unsupervised) dan dari peta 4-kelas `data/digitasi_kamal/klasifikasi_kamal.tif` (sawah/vegetasi/terbangun/air). Karena kelas air/tambak tak terwakili di training, area tambak/air pada peta biner tidak dapat dipercaya.

In [4]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

if df_feat is None:
    print("STATUS: BELUM TERSEDIA — peta dilewati.")
else:
    import joblib
    clf = joblib.load(f"{OUT}/sawah_rf.joblib")
    src = rasterio.open(RASTER)
    arr = src.read().astype(float)
    nodata = src.nodata
    h, w = arr.shape[1], arr.shape[2]
    flat = arr.reshape(arr.shape[0], -1).T
    ok = np.isfinite(flat).all(axis=1) & (~(flat == nodata).any(axis=1))
    out = np.full(flat.shape[0], 255, dtype=np.uint8)
    out[ok] = clf.predict(flat[ok]).astype(np.uint8)
    print(f"piksel valid: {int(ok.sum())} dari {flat.shape[0]}; fraksi sawah (valid): {float((out[ok] == 1).mean()):.3f}")
    meta = src.meta.copy()
    meta.update(count=1, dtype="uint8", nodata=255)
    with rasterio.open(f"{OUT}/sawah_classified.tif", "w", **meta) as dst:
        dst.write(out.reshape(h, w), 1)
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(np.ma.masked_equal(out.reshape(h, w), 255), vmin=0, vmax=1)
    ax.set_facecolor('black')
    ax.set_title("Prediksi sawah (1) / non-sawah (0) — Kec. Kamal")
    fig.tight_layout()
    fig.savefig("../data/images/eksperimen204/fig_sawah_preview.png", dpi=100)
    print("tersimpan: sawah_classified.tif + fig_sawah_preview.png")


piksel valid: 455377 dari 756875; fraksi sawah (valid): 0.479
tersimpan: sawah_classified.tif + fig_sawah_preview.png


### Peta prediksi (pratinjau)

![Peta prediksi sawah](../data/images/eksperimen204/fig_sawah_preview.png)


## Visualisasi EDA fitur (6 gambar)

Karakteristik 6 band Sentinel-2 pada 100 poligon sampel: boxplot per band per kelas, tanda tangan spektral rata-rata +- std, histogram per band, matriks korelasi antar band, PCA 2D, dan sebaran NDVI per kelas. File: `fig_sawah_boxplot.png`, `fig_sawah_spektral.png`, `fig_sawah_hist.png`, `fig_sawah_korelasi.png`, `fig_sawah_pca.png`, `fig_sawah_ndvi_violin.png` di `data/images/eksperimen204/`.

In [5]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

BANDS = ['B02', 'B03', 'B04', 'B08', 'B11', 'B12']
BCOLS = ['band_1', 'band_2', 'band_3', 'band_4', 'band_5', 'band_6']
IMG = '../data/images/eksperimen204/'

if df_feat is None:
    print('STATUS: BELUM TERSEDIA — visualisasi EDA dilewati.')
else:
    sw = df_feat[df_feat['kelas'] == 'sawah']
    ns = df_feat[df_feat['kelas'] == 'non_sawah']
    fig, axes = plt.subplots(2, 3, figsize=(12, 7))
    for ax, b, nama in zip(axes.flat, BCOLS, BANDS):
        ax.boxplot([ns[b].to_numpy(), sw[b].to_numpy()])
        ax.set_xticklabels(['non_sawah', 'sawah'])
        ax.set_title(nama)
    fig.suptitle('Sebaran nilai band per kelas (rata-rata poligon, DN Sentinel-2)')
    fig.tight_layout()
    fig.savefig(IMG + 'fig_sawah_boxplot.png', dpi=100)
    plt.close(fig)
    fig, ax = plt.subplots(figsize=(9, 5))
    for g, lab, mk in [(sw, 'sawah', 'o'), (ns, 'non_sawah', 's')]:
        m = g[BCOLS].mean().to_numpy()
        s = g[BCOLS].std().to_numpy()
        ax.errorbar(BANDS, m, yerr=s, marker=mk, capsize=4, label=lab)
    ax.set_title('Tanda tangan spektral rata-rata per kelas')
    ax.set_ylabel('DN rata-rata poligon')
    ax.legend()
    fig.tight_layout()
    fig.savefig(IMG + 'fig_sawah_spektral.png', dpi=100)
    plt.close(fig)
    fig, axes = plt.subplots(2, 3, figsize=(12, 7))
    for ax, b, nama in zip(axes.flat, BCOLS, BANDS):
        ax.hist(ns[b], bins=15, alpha=0.6, label='non_sawah')
        ax.hist(sw[b], bins=15, alpha=0.6, label='sawah')
        ax.set_title(nama)
    axes.flat[0].legend()
    fig.suptitle('Histogram nilai band per kelas')
    fig.tight_layout()
    fig.savefig(IMG + 'fig_sawah_hist.png', dpi=100)
    plt.close(fig)
    corr = df_feat[BCOLS].corr().to_numpy()
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(corr, vmin=-1, vmax=1)
    ax.set_xticks(range(6), BANDS)
    ax.set_yticks(range(6), BANDS)
    for i in range(6):
        for j in range(6):
            ax.text(j, i, f'{corr[i, j]:.2f}', ha='center', va='center', fontsize=8)
    ax.set_title('Korelasi antar band (semua sampel)')
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(IMG + 'fig_sawah_korelasi.png', dpi=100)
    plt.close(fig)
    Xs = StandardScaler().fit_transform(df_feat[BCOLS].to_numpy())
    P = PCA(n_components=2, random_state=SEED).fit_transform(Xs)
    print('PCA explained variance:', PCA(n_components=2, random_state=SEED).fit(Xs).explained_variance_ratio_.round(3).tolist())
    fig, ax = plt.subplots(figsize=(6, 5))
    m1 = df_feat['kelas'].to_numpy() == 'non_sawah'
    ax.scatter(P[m1, 0], P[m1, 1], alpha=0.7, label='non_sawah')
    ax.scatter(P[~m1, 0], P[~m1, 1], alpha=0.7, label='sawah')
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.set_title('PCA 2D fitur band per poligon')
    ax.legend()
    fig.tight_layout()
    fig.savefig(IMG + 'fig_sawah_pca.png', dpi=100)
    plt.close(fig)
    ndvi_sw = (sw['band_4'] - sw['band_3']) / (sw['band_4'] + sw['band_3'])
    ndvi_ns = (ns['band_4'] - ns['band_3']) / (ns['band_4'] + ns['band_3'])
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.violinplot([ndvi_ns.to_numpy(), ndvi_sw.to_numpy()], showmeans=True)
    ax.set_xticks([1, 2], ['non_sawah', 'sawah'])
    ax.set_title('Sebaran NDVI per kelas (dari rata-rata poligon)')
    fig.tight_layout()
    fig.savefig(IMG + 'fig_sawah_ndvi_violin.png', dpi=100)
    plt.close(fig)
    print('tersimpan: boxplot, spektral, hist, korelasi, pca, ndvi_violin')


PCA explained variance: [0.904, 0.069]
tersimpan: boxplot, spektral, hist, korelasi, pca, ndvi_violin


### Galeri EDA fitur

![Boxplot band per kelas](../data/images/eksperimen204/fig_sawah_boxplot.png)

![Tanda tangan spektral](../data/images/eksperimen204/fig_sawah_spektral.png)

![Histogram band](../data/images/eksperimen204/fig_sawah_hist.png)

![Korelasi band](../data/images/eksperimen204/fig_sawah_korelasi.png)

![PCA 2D](../data/images/eksperimen204/fig_sawah_pca.png)

![NDVI per kelas](../data/images/eksperimen204/fig_sawah_ndvi_violin.png)


## Visualisasi evaluasi dan peta (6 gambar)

Heatmap confusion matrix, feature importance RandomForest, komposit RGB + overlay 100 poligon sampel, peta probabilitas sawah, perbandingan peta sumber 4-kelas vs prediksi biner, dan peta NDVI Kamal. File: `fig_sawah_cm.png`, `fig_sawah_importance.png`, `fig_sawah_rgb_sampel.png`, `fig_sawah_prob.png`, `fig_sawah_banding.png`, `fig_sawah_ndvi_peta.png` di `data/images/eksperimen204/`.

In [6]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix as cm_fn
import geopandas as gpd
import rasterio

if df_feat is None:
    print('STATUS: BELUM TERSEDIA — visualisasi evaluasi/peta dilewati.')
else:
    import joblib
    clf2 = joblib.load(f'{OUT}/sawah_rf.joblib')
    X = df_feat.filter(like='band_').to_numpy()
    y = (df_feat['kelas'] == 'sawah').to_numpy().astype(int)
    _, Xte, _, yte = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
    pred = clf2.predict(Xte)
    cm = cm_fn(yte, pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm)
    ax.set_xticks([0, 1], ['pred non_sawah', 'pred sawah'])
    ax.set_yticks([0, 1], ['aktual non_sawah', 'aktual sawah'])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=14)
    ax.set_title('Confusion matrix (test 20 sampel)')
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(IMG + 'fig_sawah_cm.png', dpi=100)
    plt.close(fig)
    imp = clf2.feature_importances_
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.barh(BANDS, imp)
    ax.set_xlabel('importance')
    ax.set_title('Feature importance RandomForest')
    for i, v in enumerate(imp):
        ax.text(v, i, f' {v:.3f}', va='center')
    fig.tight_layout()
    fig.savefig(IMG + 'fig_sawah_importance.png', dpi=100)
    plt.close(fig)
    src = rasterio.open(RASTER)
    arr6 = src.read().astype(float)
    ok6 = np.isfinite(arr6).all(axis=0) & (~(arr6 == src.nodata).any(axis=0))
    rgb = np.stack([arr6[2], arr6[1], arr6[0]])
    lo, hi = np.percentile(rgb[:, ok6], [2, 98])
    rgb8 = np.clip((rgb - lo) / (hi - lo), 0, 1)
    b = src.bounds
    shp2 = gpd.read_file(SHP)
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(np.moveaxis(rgb8, 0, -1), extent=(b.left, b.right, b.bottom, b.top))
    shp2[shp2['kelas'] == 'sawah'].plot(ax=ax, facecolor='none', edgecolor='lime', linewidth=0.8, label='sawah')
    shp2[shp2['kelas'] == 'non_sawah'].plot(ax=ax, facecolor='none', edgecolor='red', linewidth=0.8, label='non_sawah')
    ax.set_title('Komposit RGB (B04/B03/B02) + 100 poligon sampel')
    ax.legend()
    fig.tight_layout()
    fig.savefig(IMG + 'fig_sawah_rgb_sampel.png', dpi=100)
    plt.close(fig)
    flat = arr6.reshape(6, -1).T
    okf = np.isfinite(flat).all(axis=1) & (~(flat == src.nodata).any(axis=1))
    proba = np.full(flat.shape[0], np.nan)
    proba[okf] = clf2.predict_proba(flat[okf])[:, 1]
    fig, ax = plt.subplots(figsize=(7, 7))
    im = ax.imshow(proba.reshape(arr6.shape[1], arr6.shape[2]), vmin=0, vmax=1)
    ax.set_title('Probabilitas sawah (RandomForest)')
    fig.colorbar(im, ax=ax, label='P(sawah)')
    fig.tight_layout()
    fig.savefig(IMG + 'fig_sawah_prob.png', dpi=100)
    plt.close(fig)
    src.close()
    k4 = rasterio.open('../data/digitasi_kamal/klasifikasi_kamal.tif').read(1)
    binmap = rasterio.open(f'{OUT}/sawah_classified.tif').read(1)
    fig, axes = plt.subplots(1, 2, figsize=(13, 6))
    cmap4 = ListedColormap(['black', 'green', 'darkgreen', 'gray', 'blue'])
    axes[0].imshow(k4, vmin=0, vmax=4, cmap=cmap4)
    axes[0].set_title('Sumber 4-kelas (0=nodata,1=sawah,2=veg,3=bangun,4=air)')
    binm = np.ma.masked_equal(binmap, 255)
    cmap_bin = plt.cm.viridis.copy()
    cmap_bin.set_bad('black')
    axes[1].imshow(binm, vmin=0, vmax=1, cmap=cmap_bin)
    axes[1].set_title('Prediksi biner RF (1=sawah, 0=non-sawah)')
    fig.suptitle('Banding peta sumber vs prediksi')
    fig.tight_layout()
    fig.savefig(IMG + 'fig_sawah_banding.png', dpi=100)
    plt.close(fig)
    b04 = arr6[2]
    b08 = arr6[3]
    with np.errstate(divide='ignore', invalid='ignore'):
        ndvi_map = (b08 - b04) / (b08 + b04)
    ndvi_map[~ok6] = np.nan
    fig, ax = plt.subplots(figsize=(7, 7))
    im = ax.imshow(ndvi_map, vmin=-1, vmax=1, cmap='RdYlGn')
    ax.set_title('NDVI Kamal (dari B08/B04)')
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(IMG + 'fig_sawah_ndvi_peta.png', dpi=100)
    plt.close(fig)
    print('tersimpan: cm, importance, rgb_sampel, prob, banding, ndvi_peta')


tersimpan: cm, importance, rgb_sampel, prob, banding, ndvi_peta


### Galeri evaluasi dan peta

![Confusion matrix](../data/images/eksperimen204/fig_sawah_cm.png)

![Feature importance](../data/images/eksperimen204/fig_sawah_importance.png)

![RGB + sampel](../data/images/eksperimen204/fig_sawah_rgb_sampel.png)

![Probabilitas sawah](../data/images/eksperimen204/fig_sawah_prob.png)

![Banding sumber vs prediksi](../data/images/eksperimen204/fig_sawah_banding.png)

![Peta NDVI](../data/images/eksperimen204/fig_sawah_ndvi_peta.png)


## Status akhir 6.1 — SELESAI (level kandidat, 2026-10-03)

- Training: 100 poligon (50 sawah + 50 non_sawah); test 20 sampel (10+10) dengan confusion matrix [[10, 0], [0, 10]], accuracy/precision/recall/F1 = 1.0, CV 5-fold mean 1.0. Nilai sempurna ini SIRCULAR (label kandidat dari raster yang sama) — jangan diklaim sebagai akurasi lapangan.
- Peta: `data/processed/sawah_classified.tif` (1 = sawah, 0 = non-sawah, 255 = nodata); 455.377 piksel valid, fraksi sawah 0,479. Pembanding sumber 4-kelas (`klasifikasi_kamal.tif`): sawah 37,5 persen area valid — model biner over-prediksi sawah sekitar 10 poin, konsisten dengan tidak adanya sampel air/tambak di training.
- File: `sawah_features.csv` (100 baris), `sawah_metrics.csv`, `sawah_rf.joblib`, `sawah_classified.tif`, `data/images/eksperimen204/fig_sawah_preview.png` + 12 PNG visualisasi EDA dan evaluasi/peta (boxplot, spektral, hist, korelasi, pca, ndvi_violin, cm, importance, rgb_sampel, prob, banding, ndvi_peta), total 13 gambar.
- Arti FP/FN: false positive = non-sawah diprediksi sawah (luas sawah over-estimasi); false negative = sawah tak terdeteksi (under-estimasi).
- Syarat pakai serius: validasi lapangan/citra resolusi tinggi + tambah sampel air/tambak + split blok spasial, lalu latih ulang.

## Unduh Berkas Klasifikasi Sawah

*   **sawah_features.csv** — Fitur mean band 100 poligon (50 sawah + 50 non-sawah):  
    <a href="../data/processed/sawah_features.csv" download style="display:inline-block;padding:8px 18px;margin:4px 6px 4px 0;background:#2e7d32;color:#ffffff;text-decoration:none;border-radius:8px;font-weight:bold;font-size:14px;">Unduh sawah_features.csv</a>

*   **sawah_metrics.csv** — Metrik evaluasi model:  
    <a href="../data/processed/sawah_metrics.csv" download style="display:inline-block;padding:8px 18px;margin:4px 6px 4px 0;background:#1a73e8;color:#ffffff;text-decoration:none;border-radius:8px;font-weight:bold;font-size:14px;">Unduh sawah_metrics.csv</a>

*   **sawah_classified.tif** — Raster terklasifikasi (1=sawah, 0=non-sawah):  
    <a href="../data/processed/sawah_classified.tif" download style="display:inline-block;padding:8px 18px;margin:4px 6px 4px 0;background:#1a73e8;color:#ffffff;text-decoration:none;border-radius:8px;font-weight:bold;font-size:14px;">Unduh sawah_classified.tif</a>

*   **sawah_rf.joblib** — Model RandomForest terlatih:  
    <a href="../data/processed/sawah_rf.joblib" download style="display:inline-block;padding:8px 18px;margin:4px 6px 4px 0;background:#1a73e8;color:#ffffff;text-decoration:none;border-radius:8px;font-weight:bold;font-size:14px;">Unduh sawah_rf.joblib</a>

*   **kamal_sawah.shp** — Shapefile sampel (.shp — unduh bersama 4 sidecar di bawahnya):  
    <a href="../data/spasial/kamal_sawah.shp" download style="display:inline-block;padding:8px 18px;margin:4px 6px 4px 0;background:#1a73e8;color:#ffffff;text-decoration:none;border-radius:8px;font-weight:bold;font-size:14px;">Unduh kamal_sawah.shp</a>

*   **kamal_sawah.dbf** — Sidecar .dbf:  
    <a href="../data/spasial/kamal_sawah.dbf" download style="display:inline-block;padding:8px 18px;margin:4px 6px 4px 0;background:#1a73e8;color:#ffffff;text-decoration:none;border-radius:8px;font-weight:bold;font-size:14px;">Unduh kamal_sawah.dbf</a>

*   **kamal_sawah.shx** — Sidecar .shx:  
    <a href="../data/spasial/kamal_sawah.shx" download style="display:inline-block;padding:8px 18px;margin:4px 6px 4px 0;background:#1a73e8;color:#ffffff;text-decoration:none;border-radius:8px;font-weight:bold;font-size:14px;">Unduh kamal_sawah.shx</a>

*   **kamal_sawah.prj** — Sidecar .prj:  
    <a href="../data/spasial/kamal_sawah.prj" download style="display:inline-block;padding:8px 18px;margin:4px 6px 4px 0;background:#1a73e8;color:#ffffff;text-decoration:none;border-radius:8px;font-weight:bold;font-size:14px;">Unduh kamal_sawah.prj</a>

*   **kamal_sawah.cpg** — Sidecar .cpg:  
    <a href="../data/spasial/kamal_sawah.cpg" download style="display:inline-block;padding:8px 18px;margin:4px 6px 4px 0;background:#1a73e8;color:#ffffff;text-decoration:none;border-radius:8px;font-weight:bold;font-size:14px;">Unduh kamal_sawah.cpg</a>

*   **s2_kamal.tif** — Citra Sentinel-2 Kamal (3,3 MB):  
    <a href="../data/spasial/s2_kamal.tif" download style="display:inline-block;padding:8px 18px;margin:4px 6px 4px 0;background:#1a73e8;color:#ffffff;text-decoration:none;border-radius:8px;font-weight:bold;font-size:14px;">Unduh s2_kamal.tif</a>
